# 07 — Deep Learning Anomaly Detection

## 1. Objective

This notebook develops and examines deep-learning models for unsupervised anomaly detection in the Secure Water Treatment (SWaT) dataset.

The analysis builds on the data preparation, chronological partitioning, and LSTM Autoencoder work completed in the preceding notebooks. It focuses on two additional architectures: a Temporal Convolutional Network Autoencoder (TCN Autoencoder) and Unsupervised Anomaly Detection on Multivariate Time Series (USAD).

The objective is to investigate whether these architectures can learn patterns from normal industrial process observations and assign higher anomaly scores to observations that differ from those learned patterns.

## 2. Scope

This notebook covers:

- TCN Autoencoder methodology, implementation, training, and reconstruction-score analysis.
- USAD methodology, implementation, training, and anomaly-score analysis.
- Documentation of model configurations, training behaviour, computational requirements, and observed limitations.
- Preservation and verification of the resulting model artifacts and relevant outputs.

The LSTM Autoencoder developed in Notebook 05 remains part of the project's deep-learning model set. Its existing implementation and artifacts will be reused where appropriate rather than unnecessarily reproduced.

## 3. Methodological Principles

All implementations must follow the project's established data and evaluation protocol.

- Train the models using the designated normal training partition.
- Preserve chronological separation between training, threshold-calibration, and validation observations.
- Reuse the established 42-feature definition and training-fitted scaler where appropriate.
- Do not use attack labels to fit models, select model configurations, or determine anomaly thresholds.
- Record actual training outputs, score distributions, and computational costs without inventing results.
- Document observations and limitations before proceeding to the next stage.
- Keep final threshold selection and formal evaluation in Notebook 08.

## 4. Expected Outcome

The notebook will provide reproducible implementations and documented results for the TCN Autoencoder and USAD. Together with the existing LSTM Autoencoder, these models will form the project's deep-learning detector set for subsequent threshold calibration, comparative evaluation, and error analysis.

## 5. TCN Autoencoder — Methodological Foundations

### 5.1 Objective

The Temporal Convolutional Network Autoencoder (TCN Autoencoder) is an unsupervised deep-learning model that learns to reconstruct multivariate time-series windows from normal industrial process observations.

Unlike the LSTM Autoencoder, which uses recurrent layers to represent temporal dependencies, the TCN Autoencoder uses temporal convolutions. Dilated convolutions allow the network to incorporate information from a wider temporal context without requiring a recurrent operation at every timestep.

The model will be trained using the designated normal training partition. Its reconstruction errors will provide anomaly-oriented scores for subsequent analysis.

### 5.2 Input Representation

Each model input is a multivariate time-series window:

$$
X \in \mathbb{R}^{W \times F}
$$

where:

- $W = 30$ is the established sequence-window length.
- $F = 42$ is the number of selected sensor and actuator features.

A batch of windows is represented as:

$$
X \in \mathbb{R}^{B \times W \times F}
$$

where $B$ is the batch size.

The implementation must account for PyTorch's expected input layout for one-dimensional convolutions, which is typically:

$$
X \in \mathbb{R}^{B \times F \times W}
$$

Any necessary dimension permutation must be explicit and verified.

### 5.3 Temporal Convolutions and Dilation

A temporal convolution applies learnable filters across the time dimension. Dilation introduces gaps between filter positions, allowing a convolutional layer to access a wider temporal range without proportionally increasing the kernel size.

For a one-dimensional dilated causal convolution, the operation is:

$$
y_t = \sum_{i=0}^{K-1} w_i \, x_{t - d \cdot i}
$$

where:

- $x_t$ is the input sequence.
- $y_t$ is the output at timestep $t$.
- $w_i$ represents a learnable filter coefficient.
- $K$ is the kernel size.
- $d$ is the dilation factor.

The convolution is causal when the input is padded with $(K-1) \cdot d$ zeros on its left edge and the padded sequence is indexed from $1$; equivalently, $x_p = 0$ for all $p \leq 0$. Under that padding, the output $y_t$ depends only on $x_t$ and the preceding positions $x_{t-d}, x_{t-2d}, \ldots, x_{t-(K-1)d}$. No future observation $x_{t+j}$ with $j > 0$ contributes to $y_t$.

### 5.4 Receptive Field

The receptive field is the number of input timesteps that can influence an output position.

For a stack of $L$ dilated convolutional layers with common kernel size $K$ and dilations $d_1, d_2, \ldots, d_L$, the receptive field is:

$$
R = 1 + (K-1) \sum_{\ell=1}^{L} d_\ell
$$

$L$ in this formula is the total number of convolutional layers, not the number of TCN blocks. If a block contains two convolutions with the same dilation factor, its contribution to the sum is $2 \cdot d$ and to $L$ is $2$. This distinction matters when comparing against published configurations, which typically stack 2 convolutions per block.

For the reference configuration with four TCN blocks, dilation schedule $1, 2, 4, 8$, and kernel size $3$:

- If each block contains 1 convolution ($L = 4$, $d = [1, 2, 4, 8]$): $R = 1 + 2 \cdot (1+2+4+8) = 31$.
- If each block contains 2 convolutions ($L = 8$, $d = [1,1,2,2,4,4,8,8]$): $R = 1 + 2 \cdot 30 = 61$.
- If each block contains 3 convolutions ($L = 12$, $d = [1,1,1,2,2,2,4,4,4,8,8,8]$): $R = 1 + 2 \cdot 45 = 91$.

The reference reports a receptive field of 90 timesteps. That value is consistent with 3 convolutions per block (giving 91) but not with 1 or 2; it must be verified against the actual implementation, not inferred from the dilation schedule alone. The count reported here assumes no stride, no downsampling between blocks, and no extra conv layers on the residual path — any of those would change the effective receptive field.

Because this project's established input window contains 30 timesteps, the implementation must also be checked to confirm how its receptive field compares with the window length. If the computed receptive field exceeds 30, the network's output at early window positions will be computed from padded input only, and its reconstruction of those positions may be less informative than the middle and end of the window.

### 5.5 Encoder and Decoder

The encoder transforms the input sequence into a more compact representation of its temporal patterns. Dilated convolutional layers allow the encoder to represent dependencies across multiple temporal scales.

The decoder maps the encoded representation back to the original sequence dimensions.

For an input window $X$, the reconstructed output is denoted by:

$$
\hat{X} = f_{\theta}(X)
$$

where $f_{\theta}$ represents the trainable encoder–decoder network.

The reconstructed output must preserve the input window's temporal and feature dimensions:

$$
\hat{X} \in \mathbb{R}^{B \times W \times F}
$$

This shape requirement will be verified before training.

### 5.6 Reconstruction Loss

The model uses mean squared error (MSE) as its reconstruction objective:

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{B W F}
\sum_{b=1}^{B}
\sum_{t=1}^{W}
\sum_{f=1}^{F}
\left( X_{b,t,f} - \hat{X}_{b,t,f} \right)^2
$$

where:

- $X_{b,t,f}$ is the observed feature value.
- $\hat{X}_{b,t,f}$ is its reconstructed value.
- $B$, $W$, and $F$ denote batch size, window length, and feature count.

Minimizing this loss encourages the model to reconstruct the patterns represented in its training data.

### 5.7 Anomaly Scoring

After training, reconstruction error is calculated for each evaluated sequence.

For sequence $i$, the mean squared reconstruction error is:

$$
s_i =
\frac{1}{W F}
\sum_{t=1}^{W}
\sum_{f=1}^{F}
\left( X_{i,t,f} - \hat{X}_{i,t,f} \right)^2
$$

Higher scores indicate greater reconstruction error and are interpreted as greater anomalousness under this scoring convention.

High reconstruction error does not by itself establish a physical fault or cyberattack. Formal classifications will depend on the threshold-calibration and evaluation protocol defined in Notebook 08.

### 5.8 Training and Evaluation Controls

The implementation preserves the project's existing methodological controls:

- Use the established normal training partition for model fitting.
- Reuse the existing 42-feature definition and training-fitted scaler.
- Preserve chronological separation between training, threshold-calibration, and validation data.
- Avoid using attack labels for model fitting, architecture selection, or threshold selection.
- Verify model input and output dimensions before training.
- Record the actual configuration, training history, runtime, and reconstruction-score distributions.
- Save the fitted model and verify that it can be reloaded.
- Defer final threshold selection and formal evaluation to Notebook 08.

The TCN Autoencoder will be assessed using recorded evidence from this implementation. Results from the reference notebook will not be assumed to transfer to the current dataset or configuration.

In [1]:
# 6. TCN AUTOENCODER — REQUIRED ARTIFACT VERIFICATION

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ARTIFACT_DIR = Path("/content/drive/MyDrive/swat-artifacts")
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"

if not ARTIFACT_DIR.is_dir():
    raise FileNotFoundError(
        f"Artifact directory not found: {ARTIFACT_DIR}"
    )

if not SCALER_PATH.is_file():
    raise FileNotFoundError(
        f"Training-fitted scaler not found: {SCALER_PATH}"
    )

print("Artifact verification: PASS")
print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"Scaler file: {SCALER_PATH}")
print(f"Scaler size: {SCALER_PATH.stat().st_size / (1024 ** 2):.2f} MB")

Mounted at /content/drive
Artifact verification: PASS
Artifact directory: /content/drive/MyDrive/swat-artifacts
Scaler file: /content/drive/MyDrive/swat-artifacts/scaler.joblib
Scaler size: 0.00 MB


In [2]:
# 6.1 LOAD AND VERIFY THE EXISTING SCALER

import joblib
import numpy as np

# Load the scaler fitted on the designated training partition.
scaler = joblib.load(SCALER_PATH)

# Recover the established feature names and their ordering.
if not hasattr(scaler, "feature_names_in_"):
    raise AttributeError(
        "The saved scaler does not contain feature_names_in_. "
        "The established feature ordering must be verified separately."
    )

FEATURE_COLUMNS = list(scaler.feature_names_in_)

# Verify the expected feature count.
assert len(FEATURE_COLUMNS) == 42, (
    f"Expected 42 features, found {len(FEATURE_COLUMNS)}."
)

# Verify feature names are unique.
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)), (
    "Duplicate feature names detected."
)

# Verify the scaler's fitted dimensionality.
assert scaler.n_features_in_ == len(FEATURE_COLUMNS), (
    "Scaler feature count does not match its recorded feature names."
)

print("Scaler verification: PASS")
print(f"Scaler type: {type(scaler).__name__}")
print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"Fitted feature count: {scaler.n_features_in_}")
print(f"Feature names and order: {FEATURE_COLUMNS}")

Scaler verification: PASS
Scaler type: MinMaxScaler
Number of features: 42
Fitted feature count: 42
Feature names and order: ['FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P203', 'P205', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P402', 'P403', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P602']


In [3]:
from pathlib import Path
import json

METADATA_PATH = ARTIFACT_DIR / "traditional_ml_model_metadata.json"

print(f"Metadata file exists: {METADATA_PATH.is_file()}")

if METADATA_PATH.is_file():
    with open(METADATA_PATH, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    print("Metadata type:", type(metadata).__name__)

    if isinstance(metadata, dict):
        print("Available metadata keys:")
        for key in metadata:
            print(f"- {key}")
else:
    print("Traditional ML metadata is unavailable; it is not required for TCN training.")

Metadata file exists: True
Metadata type: dict
Available metadata keys:
- notebook
- scikit_learn_version
- feature_count
- feature_columns
- training_observations
- training_features
- models


In [4]:
for path in sorted(ARTIFACT_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path.name:<45} {size_mb:>10.2f} MB")
    elif path.is_dir():
        print(f"{path.name}/")

evaluation_sequences.npz                           10.24 MB
isolation_forest.joblib                             2.27 MB
lof_model.joblib                                   54.05 MB
lstm_autoencoder.pth                                0.51 MB
scaler.joblib                                       0.00 MB
sgd_one_class_svm.joblib                            0.00 MB
tcn_preliminary/
traditional_ml_model_metadata.json                  0.00 MB


In [5]:
from pathlib import Path

NORMAL_PATH = Path("/content/drive/MyDrive/swat-dataset/normal.csv")

if NORMAL_PATH.is_file():
    print("Normal dataset: FOUND")
    print(f"Path: {NORMAL_PATH}")
    print(f"Size: {NORMAL_PATH.stat().st_size / (1024 ** 2):.2f} MB")
else:
    print("Normal dataset: NOT FOUND")
    print(f"Expected path: {NORMAL_PATH}")

Normal dataset: FOUND
Path: /content/drive/MyDrive/swat-dataset/normal.csv
Size: 383.73 MB


In [6]:
import numpy as np

SEQUENCE_PATH = ARTIFACT_DIR / "evaluation_sequences.npz"

with np.load(SEQUENCE_PATH, allow_pickle=False) as sequences:
    print("Saved sequence arrays:")

    for name in sequences.files:
        array = sequences[name]
        print(
            f"{name}: "
            f"shape={array.shape}, "
            f"dtype={array.dtype}, "
            f"finite={np.isfinite(array).all()}"
        )

Saved sequence arrays:
X_threshold_seq: shape=(47408, 30, 42), dtype=float32, finite=True
X_validation_seq: shape=(79031, 30, 42), dtype=float32, finite=True
threshold_end_indices: shape=(47408,), dtype=int64, finite=True
validation_end_indices: shape=(79031,), dtype=int64, finite=True


In [7]:
with np.load(SEQUENCE_PATH, allow_pickle=False) as saved:
    threshold_indices = saved["threshold_end_indices"].copy()
    validation_indices = saved["validation_end_indices"].copy()

for name, indices in (
    ("threshold_end_indices", threshold_indices),
    ("validation_end_indices", validation_indices),
):
    print(f"\n{name}")
    print(f"Count: {len(indices):,}")
    print(f"First index: {indices[0]:,}")
    print(f"Last index: {indices[-1]:,}")
    print(f"Strictly increasing: {np.all(np.diff(indices) > 0)}")


threshold_end_indices
Count: 47,408
First index: 29
Last index: 47,436
Strictly increasing: True

validation_end_indices
Count: 79,031
First index: 29
Last index: 79,059
Strictly increasing: True


In [8]:
print("Notebook:", metadata.get("notebook"))
print("Recorded feature count:", metadata.get("feature_count"))
print("Recorded training observations:", metadata.get("training_observations"))

recorded_features = metadata.get("feature_columns", [])

print("Recorded feature list length:", len(recorded_features))
print("Matches scaler feature order:", recorded_features == FEATURE_COLUMNS)

Notebook: 06_traditional_ml_anomaly_detection
Recorded feature count: 42
Recorded training observations: 221366
Recorded feature list length: 42
Matches scaler feature order: True


In [9]:
import pandas as pd
import numpy as np
import warnings

TRAIN_ROWS = 221_366
MONITOR_ROWS = 47_435
THRESHOLD_ROWS = 47_437
VALIDATION_ROWS = 79_060

ROWS_NEEDED = TRAIN_ROWS + MONITOR_ROWS
TIMESTAMP_COL = "Timestamp"
CHUNK_SIZE = 100_000

warnings.filterwarnings("ignore", message=".*Parsing dates.*")

# Read the header only to identify the sensor columns.
normal_header = pd.read_csv(NORMAL_PATH, nrows=0).columns.str.strip().tolist()

if TIMESTAMP_COL not in normal_header:
    raise KeyError("The normal dataset does not contain 'Timestamp'.")

LABEL_COL = next(
    (
        column
        for column in normal_header
        if column.lower() in {"normal/attack", "label"}
    ),
    None,
)

if LABEL_COL is None:
    raise KeyError("The normal dataset's label column was not found.")

SENSOR_COLUMNS = [
    column
    for column in normal_header
    if column not in {TIMESTAMP_COL, LABEL_COL}
]

if not set(FEATURE_COLUMNS).issubset(SENSOR_COLUMNS):
    raise ValueError("The scaler's 42 features do not match the CSV schema.")

eligible_parts = []
eligible_rows_collected = 0

for chunk in pd.read_csv(
    NORMAL_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False,
):
    chunk.columns = chunk.columns.str.strip()

    # Parse timestamps and retain only valid, complete observations.
    chunk[TIMESTAMP_COL] = pd.to_datetime(
        chunk[TIMESTAMP_COL],
        errors="coerce",
    )

    valid_mask = (
        chunk[TIMESTAMP_COL].notna()
        & chunk[SENSOR_COLUMNS].notna().all(axis=1)
    )

    eligible_chunk = chunk.loc[valid_mask].copy()

    if eligible_chunk.empty:
        continue

    # Retain only as many eligible rows as are still required.
    rows_remaining = ROWS_NEEDED - eligible_rows_collected
    eligible_chunk = eligible_chunk.iloc[:rows_remaining]

    eligible_parts.append(
        eligible_chunk[[TIMESTAMP_COL] + SENSOR_COLUMNS]
    )

    eligible_rows_collected += len(eligible_chunk)

    if eligible_rows_collected >= ROWS_NEEDED:
        break

if eligible_rows_collected != ROWS_NEEDED:
    raise ValueError(
        f"Expected {ROWS_NEEDED:,} eligible observations, "
        f"but recovered {eligible_rows_collected:,}."
    )

# Combine only the rows needed for training and monitoring.
training_monitor_df = pd.concat(
    eligible_parts,
    ignore_index=True,
)

# Confirm that the reconstructed rows preserve chronological order.
if not training_monitor_df[TIMESTAMP_COL].is_monotonic_increasing:
    raise ValueError(
        "The reconstructed timestamps are not monotonically increasing. "
        "Stop and investigate before training."
    )

# Verify completeness across all sensor columns.
if training_monitor_df[SENSOR_COLUMNS].isna().any().any():
    raise ValueError("Missing sensor values remain in the reconstructed data.")

# Create the established chronological partitions.
training_df = training_monitor_df.iloc[:TRAIN_ROWS].copy()
monitor_df = training_monitor_df.iloc[TRAIN_ROWS:].copy()

assert len(training_df) == TRAIN_ROWS
assert len(monitor_df) == MONITOR_ROWS

print("Training and monitoring data reconstruction: PASS")
print(f"Eligible observations recovered: {len(training_monitor_df):,}")
print(f"Training observations: {len(training_df):,}")
print(f"Monitoring observations: {len(monitor_df):,}")
print(f"All sensor columns: {len(SENSOR_COLUMNS)}")
print(f"Selected model features: {len(FEATURE_COLUMNS)}")

print(
    f"Training period: {training_df[TIMESTAMP_COL].iloc[0]} "
    f"to {training_df[TIMESTAMP_COL].iloc[-1]}"
)
print(
    f"Monitoring period: {monitor_df[TIMESTAMP_COL].iloc[0]} "
    f"to {monitor_df[TIMESTAMP_COL].iloc[-1]}"
)

print(
    "Training precedes monitoring:",
    training_df[TIMESTAMP_COL].iloc[-1]
    < monitor_df[TIMESTAMP_COL].iloc[0],
)

/tmp/ipykernel_1617/1527631077.py:54: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk[TIMESTAMP_COL] = pd.to_datetime(
/tmp/ipykernel_1617/1527631077.py:54: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk[TIMESTAMP_COL] = pd.to_datetime(


Training and monitoring data reconstruction: PASS
Eligible observations recovered: 268,801
Training observations: 221,366
Monitoring observations: 47,435
All sensor columns: 51
Selected model features: 42
Training period: 2015-12-28 10:00:00 to 2015-12-31 12:43:42
Monitoring period: 2015-12-31 12:43:43 to 2016-01-01 02:23:17
Training precedes monitoring: True


In [10]:
# Select the established 42 features in the scaler's recorded order.
X_train_raw = training_df[FEATURE_COLUMNS]
X_monitor_raw = monitor_df[FEATURE_COLUMNS]

# Transform using the existing training-fitted scaler.
X_train = scaler.transform(X_train_raw).astype(np.float32)
X_monitor = scaler.transform(X_monitor_raw).astype(np.float32)

# Verify shapes and numerical validity.
assert X_train.shape == (TRAIN_ROWS, len(FEATURE_COLUMNS))
assert X_monitor.shape == (MONITOR_ROWS, len(FEATURE_COLUMNS))
assert np.isfinite(X_train).all()
assert np.isfinite(X_monitor).all()

print("TCN feature scaling: PASS")
print(f"Training matrix: {X_train.shape}, dtype={X_train.dtype}")
print(f"Monitoring matrix: {X_monitor.shape}, dtype={X_monitor.dtype}")
print(f"Training values finite: {np.isfinite(X_train).all()}")
print(f"Monitoring values finite: {np.isfinite(X_monitor).all()}")
print(f"Training range: {X_train.min():.6f} to {X_train.max():.6f}")
print(f"Monitoring range: {X_monitor.min():.6f} to {X_monitor.max():.6f}")

TCN feature scaling: PASS
Training matrix: (221366, 42), dtype=float32
Monitoring matrix: (47435, 42), dtype=float32
Training values finite: True
Monitoring values finite: True
Training range: -0.000074 to 1.000015
Monitoring range: -0.564154 to 1.024334


In [11]:
WINDOW_SIZE = 30

def create_sequences(data, window_size):
    if data.ndim != 2:
        raise ValueError("Expected a 2D array: observations × features.")

    if len(data) < window_size:
        raise ValueError("Not enough observations to create one sequence.")

    sequences = np.lib.stride_tricks.sliding_window_view(
        data,
        window_shape=window_size,
        axis=0,
    )

    # sliding_window_view returns (windows, features, timesteps).
    # Convert explicitly to (windows, timesteps, features).
    sequences = np.moveaxis(sequences, -1, 1)

    return np.ascontiguousarray(sequences, dtype=np.float32)


X_train_seq = create_sequences(X_train, WINDOW_SIZE)
X_monitor_seq = create_sequences(X_monitor, WINDOW_SIZE)

assert X_train_seq.shape == (
    TRAIN_ROWS - WINDOW_SIZE + 1,
    WINDOW_SIZE,
    len(FEATURE_COLUMNS),
)

assert X_monitor_seq.shape == (
    MONITOR_ROWS - WINDOW_SIZE + 1,
    WINDOW_SIZE,
    len(FEATURE_COLUMNS),
)

assert np.isfinite(X_train_seq).all()
assert np.isfinite(X_monitor_seq).all()

# Verify that the first and last windows match their source matrices.
np.testing.assert_array_equal(X_train_seq[0], X_train[:WINDOW_SIZE])
np.testing.assert_array_equal(X_train_seq[-1], X_train[-WINDOW_SIZE:])
np.testing.assert_array_equal(X_monitor_seq[0], X_monitor[:WINDOW_SIZE])
np.testing.assert_array_equal(X_monitor_seq[-1], X_monitor[-WINDOW_SIZE:])

print("TCN sequence creation: PASS")
print(f"Training sequences: {X_train_seq.shape}")
print(f"Monitoring sequences: {X_monitor_seq.shape}")
print(f"Training sequence dtype: {X_train_seq.dtype}")
print(f"Monitoring sequence dtype: {X_monitor_seq.dtype}")
print(f"All training sequences finite: {np.isfinite(X_train_seq).all()}")
print(f"All monitoring sequences finite: {np.isfinite(X_monitor_seq).all()}")

TCN sequence creation: PASS
Training sequences: (221337, 30, 42)
Monitoring sequences: (47406, 30, 42)
Training sequence dtype: float32
Monitoring sequence dtype: float32
All training sequences finite: True
All monitoring sequences finite: True


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_FEATURES = len(FEATURE_COLUMNS)
WINDOW_SIZE = 30
CHANNELS = [32, 32, 32, 32]
KERNEL_SIZE = 3
DILATIONS = [1, 2, 4, 8]


class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0,
        )

    def forward(self, x):
        return self.conv(F.pad(x, (self.left_padding, 0)))


class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()

        self.conv1 = CausalConv1d(
            in_channels, out_channels, kernel_size, dilation
        )
        self.conv2 = CausalConv1d(
            out_channels, out_channels, kernel_size, dilation
        )

        self.activation = nn.ReLU()
        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(in_channels, out_channels, kernel_size=1)
        )

    def forward(self, x):
        residual = self.residual(x)
        x = self.activation(self.conv1(x))
        x = self.activation(self.conv2(x))
        return self.activation(x + residual)


class TCNAutoencoder(nn.Module):
    def __init__(self, n_features, channels, kernel_size, dilations):
        super().__init__()

        if len(channels) != len(dilations):
            raise ValueError("Each TCN block must have a dilation value.")

        encoder_layers = []
        in_channels = n_features

        for out_channels, dilation in zip(channels, dilations):
            encoder_layers.append(
                TCNBlock(
                    in_channels,
                    out_channels,
                    kernel_size,
                    dilation,
                )
            )
            in_channels = out_channels

        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        reversed_channels = list(reversed(channels))
        reversed_dilations = list(reversed(dilations))

        for index, dilation in enumerate(reversed_dilations):
            current_channels = reversed_channels[index]
            next_channels = (
                reversed_channels[index + 1]
                if index + 1 < len(reversed_channels)
                else n_features
            )

            decoder_layers.append(
                CausalConv1d(
                    current_channels,
                    next_channels,
                    kernel_size,
                    dilation,
                )
            )

            if index + 1 < len(reversed_dilations):
                decoder_layers.append(nn.ReLU())

        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(
                "Expected input shape (batch, timesteps, features)."
            )

        x = x.transpose(1, 2).contiguous()
        x = self.encoder(x)
        x = self.decoder(x)

        return x.transpose(1, 2).contiguous()


tcn_model = TCNAutoencoder(
    n_features=N_FEATURES,
    channels=CHANNELS,
    kernel_size=KERNEL_SIZE,
    dilations=DILATIONS,
).to(DEVICE)

test_input = torch.from_numpy(X_train_seq[:8]).to(DEVICE)

with torch.no_grad():
    test_output = tcn_model(test_input)

receptive_field = 1 + sum(
    2 * (KERNEL_SIZE - 1) * dilation
    for dilation in DILATIONS
)

parameter_count = sum(
    parameter.numel()
    for parameter in tcn_model.parameters()
    if parameter.requires_grad
)

assert test_output.shape == test_input.shape
assert torch.isfinite(test_output).all()
assert receptive_field == 61

print("TCN architecture verification: PASS")
print(f"Device: {DEVICE}")
print(f"Channels: {CHANNELS}")
print(f"Dilations: {DILATIONS}")
print(f"Convolutions per block: 2")
print(f"Receptive field: {receptive_field} timesteps")
print(f"Input shape: {tuple(test_input.shape)}")
print(f"Output shape: {tuple(test_output.shape)}")
print(f"Trainable parameters: {parameter_count:,}")
print(f"Finite outputs: {torch.isfinite(test_output).all().item()}")

TCN architecture verification: PASS
Device: cpu
Channels: [32, 32, 32, 32]
Dilations: [1, 2, 4, 8]
Convolutions per block: 2
Receptive field: 61 timesteps
Input shape: (8, 30, 42)
Output shape: (8, 30, 42)
Trainable parameters: 40,554
Finite outputs: True


## 7.1 Architecture Configuration and Verification

The TCN Autoencoder uses four residual convolutional blocks, each containing two causal convolutions. The kernel size is 3, the channel configuration is `[32, 32, 32, 32]`, and the dilation schedule is `[1, 2, 4, 8]`.

This configuration uses exponentially increasing dilations to expand the theoretical temporal receptive field. Its suitability will be assessed through training behaviour and reconstruction-score analysis rather than assumed in advance.

### Architecture Configuration

| Parameter | Value |
|---|---|
| Input window length | 30 timesteps |
| Input features | 42 |
| Convolutional blocks | 4 |
| Convolutions per block | 2 |
| Channels | `[32, 32, 32, 32]` |
| Kernel size | 3 |
| Dilation schedule | `[1, 2, 4, 8]` |
| Residual connections | Yes |
| Padding | Left-only causal padding |
| Trainable parameters | 40,554 |
| Input and output shape | `(batch, 30, 42)` |

### Receptive Field

The receptive field is the number of input timesteps that can influence an output position through the encoder's convolutional stack.

For causal convolutional layers with stride 1:

$$
R = 1 + \sum_{\ell=1}^{L}(K_\ell-1)d_\ell
$$

where $K_\ell$ is the kernel size and $d_\ell$ is the dilation of layer $\ell$.

Each residual block contains two convolutions with the same dilation. For four blocks with kernel size 3 and dilations 1, 2, 4, and 8:

$$
R = 1 + 2(3-1)(1+2+4+8)
$$

$$
R = 61
$$

The encoder's theoretical receptive field is 61 timesteps, exceeding the 30-timestep input window.

This does not mean that the model receives 61 actual observations. Each input window contains only 30 timesteps. Where the theoretical receptive field extends beyond the available window, left padding supplies the missing context.

The larger theoretical receptive field is an architectural design choice, not evidence of improved reconstruction or anomaly-detection performance. Its suitability will be assessed using the recorded training and evaluation results.

### Tensor Layout and Shape Verification

Input sequences are represented as:

$$
X \in \mathbb{R}^{B \times W \times F}
$$

where $B$ is batch size, $W=30$ is window length, and $F=42$ is the number of features.

PyTorch's one-dimensional convolution layers expect channel-first tensors:

$$
X \in \mathbb{R}^{B \times F \times W}
$$

The implementation explicitly permutes the input before convolution and restores the original layout afterward.

The architecture verification passed with an input shape of `(8, 30, 42)` and an output shape of `(8, 30, 42)`. All output values were finite. The verified implementation contains 40,554 trainable parameters.

### Design Considerations

Causal convolutions ensure that each convolutional output position depends only on the current and preceding positions within the input window. Left-only padding preserves the temporal length without introducing future observations into earlier convolutional outputs.

The decoder also uses causal convolutions. Successful shape verification does not establish that the architecture reconstructs all positions equally well or that the model will distinguish normal process behaviour from anomalous behaviour.

Training loss, monitoring loss, reconstruction-score distributions, and numerical stability will therefore be examined before drawing conclusions about model performance.

The four-block architecture is the configuration selected for the initial experiment. Any subsequent modification must be documented and justified by evidence.

In [13]:
print("TCN TRAINING READINESS")
print("-" * 50)

print(f"Device: {DEVICE}")
print(f"Training sequences: {len(X_train_seq):,}")
print(f"Monitoring sequences: {len(X_monitor_seq):,}")
print(f"Input shape: {tuple(X_train_seq.shape[1:])}")
print(f"Model parameters: {sum(p.numel() for p in tcn_model.parameters() if p.requires_grad):,}")

if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(DEVICE).total_memory / (1024**3):.2f} GB")
else:
    print("GPU: Not available; CPU training will be used.")

print("-" * 50)
print("Readiness inspection complete. No training started.")

TCN TRAINING READINESS
--------------------------------------------------
Device: cpu
Training sequences: 221,337
Monitoring sequences: 47,406
Input shape: (30, 42)
Model parameters: 40,554
GPU: Not available; CPU training will be used.
--------------------------------------------------
Readiness inspection complete. No training started.


## 7.2 — TCN Autoencoder Training

### Training Objective

The Temporal Convolutional Network (TCN) Autoencoder is trained to reconstruct normal sensor sequences. It learns temporal patterns in the training data by minimizing the mean squared error (MSE) between the original input sequences and their reconstructions.

For an input sequence $X$ and its reconstruction $\hat{X}$, the reconstruction loss is defined as:

$$
\mathcal{L}_{\mathrm{MSE}} =
\frac{1}{NTF}
\sum_{i=1}^{N}
\sum_{t=1}^{T}
\sum_{f=1}^{F}
\left(X_{i,t,f} - \hat{X}_{i,t,f}\right)^2
$$

where:

- $N$ is the number of sequences in the batch.
- $T$ is the number of timesteps in each sequence.
- $F$ is the number of sensor features.
- $X_{i,t,f}$ is the original value of feature $f$ at timestep $t$ in sequence $i$.
- $\hat{X}_{i,t,f}$ is the corresponding reconstructed value.

The model is optimized using the Adam optimizer with a learning rate of $0.001$ and a batch size of $256$. Training is limited to a maximum of 50 epochs, with early stopping after five consecutive epochs without improvement in monitoring loss.

### Data Separation

The training and monitoring datasets serve distinct purposes:

- **Training sequences:** Used to calculate reconstruction loss and update the model's weights through backpropagation.
- **Monitoring sequences:** Used to assess reconstruction performance during training, support early stopping, and select the best model checkpoint.
- **Threshold-calibration data:** Reserved for establishing the anomaly-detection threshold in a subsequent stage.
- **Final evaluation data:** Reserved for evaluating the selected model and its anomaly-detection performance.

Monitoring data is used to calculate loss but does not contribute gradients or optimizer updates. The best checkpoint is selected according to monitoring loss, not anomaly-detection performance on the final evaluation data.

In [14]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
required_names = [
    "X_train_seq",
    "X_monitor_seq",
    "N_FEATURES",
    "CHANNELS",
    "KERNEL_SIZE",
    "DILATIONS",
    "WINDOW_SIZE",
    "TCNAutoencoder",
]

missing_names = [
    name for name in required_names
    if name not in globals()
]

if missing_names:
    raise NameError(
        "Missing prerequisites: " + ", ".join(missing_names)
    )

if X_train_seq.ndim != 3 or X_monitor_seq.ndim != 3:
    raise ValueError(
        "Training and monitoring arrays must have shape (N, T, F)."
    )

if X_train_seq.shape[1:] != X_monitor_seq.shape[1:]:
    raise ValueError(
        "Training and monitoring sequence dimensions do not match."
    )

if X_train_seq.shape[0] == 0 or X_monitor_seq.shape[0] == 0:
    raise ValueError(
        "Training and monitoring sequences must both be non-empty."
    )

if X_train_seq.shape[2] != N_FEATURES:
    raise ValueError(
        f"Expected {N_FEATURES} features, "
        f"but found {X_train_seq.shape[2]}."
    )

if not np.isfinite(X_train_seq).all():
    raise ValueError("Training sequences contain non-finite values.")

if not np.isfinite(X_monitor_seq).all():
    raise ValueError("Monitoring sequences contain non-finite values.")

In [16]:
LEARNING_RATE = 1e-3
BATCH_SIZE = 256
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5

criterion = nn.MSELoss()

In [17]:
# TCNAutoencoder is the architecture defined in Section 7.1.
tcn_model = TCNAutoencoder(
    n_features=N_FEATURES,
    channels=CHANNELS,
    kernel_size=KERNEL_SIZE,
    dilations=DILATIONS,
).to(DEVICE)

optimizer = torch.optim.Adam(
    tcn_model.parameters(),
    lr=LEARNING_RATE,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

In [18]:
# torch.from_numpy shares the existing NumPy arrays' memory;
# sequences are moved to the device per batch.
X_train_tensor = torch.from_numpy(
    np.ascontiguousarray(X_train_seq)
)

X_monitor_tensor = torch.from_numpy(
    np.ascontiguousarray(X_monitor_seq)
)

train_dataset = TensorDataset(X_train_tensor)
monitor_dataset = TensorDataset(X_monitor_tensor)

# Dedicated generator so the training shuffle is seeded
# independently of model initialization.
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    generator=loader_generator,
)

monitor_loader = DataLoader(
    monitor_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

In [19]:
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/swat-artifacts/tcn_preliminary"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = OUTPUT_DIR / "tcn_autoencoder_best.pth"
history_path = OUTPUT_DIR / "tcn_training_history.csv"
metadata_path = OUTPUT_DIR / "tcn_training_metadata.json"
mse_plot_path = OUTPUT_DIR / "tcn_training_mse.png"
mae_plot_path = OUTPUT_DIR / "tcn_training_mae.png"

In [ ]:
best_monitor_mse = float("inf")
best_epoch = 0
epochs_without_improvement = 0
history = []

start_time = time.time()
initial_learning_rate = optimizer.param_groups[0]["lr"]

print("TCN CONTROLLED TRAINING")
print(f"Device: {DEVICE}")
print(f"Training sequences: {len(train_dataset):,}")
print(f"Monitoring sequences: {len(monitor_dataset):,}")
print(f"Sequence shape: {tuple(X_train_seq.shape[1:])}")
print(
    f"Trainable parameters: "
    f"{sum(p.numel() for p in tcn_model.parameters() if p.requires_grad):,}"
)
print(f"Seed: {SEED}")
print(f"Initial learning rate: {initial_learning_rate}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Maximum epochs: {MAX_EPOCHS}")
print(f"Early-stopping patience: {EARLY_STOPPING_PATIENCE}")

for epoch in range(1, MAX_EPOCHS + 1):

    tcn_model.train()

    train_squared_error = 0.0
    train_absolute_error = 0.0
    train_element_count = 0

    for (batch,) in train_loader:
        batch = batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        reconstruction = tcn_model(batch)
        loss = criterion(reconstruction, batch)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite training loss at epoch {epoch}."
            )

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            error = reconstruction.detach() - batch

            train_squared_error += error.square().sum().item()
            train_absolute_error += error.abs().sum().item()
            train_element_count += error.numel()

    train_mse = train_squared_error / train_element_count
    train_mae = train_absolute_error / train_element_count

    tcn_model.eval()

    monitor_squared_error = 0.0
    monitor_absolute_error = 0.0
    monitor_element_count = 0

    with torch.no_grad():
        for (batch,) in monitor_loader:
            batch = batch.to(DEVICE, non_blocking=True)

            reconstruction = tcn_model(batch)
            error = reconstruction - batch

            monitor_squared_error += error.square().sum().item()
            monitor_absolute_error += error.abs().sum().item()
            monitor_element_count += error.numel()

    monitor_mse = monitor_squared_error / monitor_element_count
    monitor_mae = monitor_absolute_error / monitor_element_count

    if not np.isfinite(monitor_mse):
        raise FloatingPointError(
            f"Non-finite monitoring MSE at epoch {epoch}."
        )

    # Checkpoint is selected on monitoring MSE only.
    improved = monitor_mse < best_monitor_mse

    if improved:
        best_monitor_mse = monitor_mse
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(tcn_model.state_dict(), checkpoint_path)
    else:
        epochs_without_improvement += 1

    scheduler.step(monitor_mse)

    current_learning_rate = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_mse": train_mse,
        "train_mae": train_mae,
        "monitor_mse": monitor_mse,
        "monitor_mae": monitor_mae,
        "learning_rate": current_learning_rate,
        "best_checkpoint_updated": improved,
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train MSE: {train_mse:.8f} | "
        f"Monitor MSE: {monitor_mse:.8f} | "
        f"Monitor MAE: {monitor_mae:.8f} | "
        f"LR: {current_learning_rate:.2e}"
        + (" | Best checkpoint" if improved else "")
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping at epoch {epoch}; "
            f"best epoch was {best_epoch}."
        )
        break

TCN CONTROLLED TRAINING
Device: cpu
Training sequences: 221,337
Monitoring sequences: 47,406
Sequence shape: (30, 42)
Trainable parameters: 40,554
Seed: 42
Initial learning rate: 0.001
Batch size: 256
Maximum epochs: 50
Early-stopping patience: 5
Epoch 01/50 | Train MSE: 0.02125573 | Monitor MSE: 0.00616963 | Monitor MAE: 0.04225124 | LR: 1.00e-03 | Best checkpoint
Epoch 02/50 | Train MSE: 0.00291673 | Monitor MSE: 0.00511769 | Monitor MAE: 0.03788232 | LR: 1.00e-03 | Best checkpoint
Epoch 03/50 | Train MSE: 0.00181649 | Monitor MSE: 0.00330128 | Monitor MAE: 0.03002732 | LR: 1.00e-03 | Best checkpoint
Epoch 04/50 | Train MSE: 0.00116977 | Monitor MSE: 0.00233433 | Monitor MAE: 0.02536222 | LR: 1.00e-03 | Best checkpoint
Epoch 05/50 | Train MSE: 0.00087200 | Monitor MSE: 0.00178662 | Monitor MAE: 0.02084928 | LR: 1.00e-03 | Best checkpoint
Epoch 06/50 | Train MSE: 0.00070018 | Monitor MSE: 0.00166127 | Monitor MAE: 0.01988952 | LR: 1.00e-03 | Best checkpoint
Epoch 07/50 | Train MSE: 0.

In [ ]:
if best_epoch == 0 or not checkpoint_path.exists():
    raise RuntimeError(
        "Training finished without producing a valid best checkpoint."
    )

try:
    best_state = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=True,
    )
except TypeError:
    # Compatibility with PyTorch versions without weights_only.
    best_state = torch.load(
        checkpoint_path,
        map_location=DEVICE,
    )

tcn_model.load_state_dict(best_state)
tcn_model.eval()

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)

elapsed_seconds = time.time() - start_time

metadata = {
    "model": "TCN Autoencoder",
    "seed": SEED,
    "device": str(DEVICE),
    "n_features": N_FEATURES,
    "window_size": WINDOW_SIZE,
    "channels": CHANNELS,
    "kernel_size": KERNEL_SIZE,
    "dilations": DILATIONS,
    "convolutions_per_block": 2,
    "receptive_field": 1 + sum(
        2 * (KERNEL_SIZE - 1) * dilation
        for dilation in DILATIONS
    ),
    "parameter_count": sum(
        parameter.numel()
        for parameter in tcn_model.parameters()
        if parameter.requires_grad
    ),
    "training_sequences": len(train_dataset),
    "monitoring_sequences": len(monitor_dataset),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "epochs_completed": len(history),
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "initial_learning_rate": initial_learning_rate,
    "best_epoch": best_epoch,
    "best_monitor_mse": best_monitor_mse,
    "best_epoch_monitor_mae": history[best_epoch - 1]["monitor_mae"],
    "elapsed_seconds": elapsed_seconds,
    "checkpoint_selection_metric": "monitor_mse",
    "checkpoint_path": str(checkpoint_path),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_mse"], label="Training MSE")
plt.plot(history_df["epoch"], history_df["monitor_mse"], label="Monitoring MSE")
plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch: {best_epoch}",
)
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("TCN Autoencoder Training and Monitoring MSE")
plt.legend()
plt.tight_layout()
plt.savefig(mse_plot_path, dpi=150)
plt.show()
plt.close()

plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_mae"], label="Training MAE")
plt.plot(history_df["epoch"], history_df["monitor_mae"], label="Monitoring MAE")
plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch: {best_epoch}",
)
plt.xlabel("Epoch")
plt.ylabel("Mean Absolute Error")
plt.title("TCN Autoencoder Training and Monitoring MAE")
plt.legend()
plt.tight_layout()
plt.savefig(mae_plot_path, dpi=150)
plt.show()
plt.close()

In [ ]:
print("\nTCN TRAINING COMPLETE")
print(f"Best epoch: {best_epoch}")
print(f"Best monitoring MSE: {best_monitor_mse:.8f}")
print(
    f"Monitoring MAE at best epoch: "
    f"{history[best_epoch - 1]['monitor_mae']:.8f}"
)
print(f"Epochs completed: {len(history)}")
print(f"Elapsed time: {elapsed_seconds / 60:.2f} minutes")
print(f"Checkpoint: {checkpoint_path}")
print(f"History: {history_path}")
print(f"Metadata: {metadata_path}")
print(f"MSE plot: {mse_plot_path}")
print(f"MAE plot: {mae_plot_path}")

### 7.3.1 — Preliminary TCN Training Results

#### Training Configuration

The TCN Autoencoder was trained for a maximum of 50 epochs using 221,337 training sequences and monitored on 47,406 chronologically subsequent sequences. Each sequence contained 30 timesteps and 42 features.

Training used mean squared error (MSE), the Adam optimizer, an initial learning rate of 0.001, and a batch size of 256. A ReduceLROnPlateau scheduler reduced the learning rate when monitoring loss stopped improving. Early stopping used a patience of five epochs.

#### Observed Results

The lowest recorded monitoring MSE was 0.00062031 at epoch 49. The corresponding training MSE was 0.00014374, and the monitoring MAE was 0.01103645.

Monitoring MSE decreased from 0.00751307 in epoch 1 to 0.00062031 in epoch 49. The learning rate was reduced twice, reaching 0.00025 by epoch 48. Training completed all 50 epochs without triggering early stopping.

#### Interpretation

The decreasing monitoring loss indicates that reconstruction performance improved on the monitoring partition during training. The difference between training and monitoring losses warrants further observation but is not sufficient, by itself, to establish harmful overfitting.

The best checkpoint was recorded at epoch 49 based on monitoring MSE. This configuration remains preliminary: the model has not yet undergone systematic hyperparameter tuning or threshold calibration.

Reconstruction loss alone does not establish anomaly-detection effectiveness. The TCN's ability to distinguish anomalous from normal observations must be assessed separately using an appropriately calibrated threshold and a verified labelled evaluation dataset.

#### Limitations and Next Steps

- This is a single preliminary TCN configuration, not a tuned or final model.
- The monitoring partition was used for checkpoint selection and is therefore not an independent final evaluation set.
- No attack-detection performance is claimed from these reconstruction results.
- Hyperparameter tuning and threshold calibration remain assigned to Notebook 08.

The TCN training results are recorded. The next step is to implement and train the USAD model within this notebook, following the established experimental protocol.

## 7.4 — USAD: UnSupervised Anomaly Detection

### 7.4.1 — Objective

USAD (UnSupervised Anomaly Detection) is an unsupervised anomaly-detection method based on autoencoders. It uses an encoder and two decoders trained through a joint optimization strategy to learn representations of normal operating patterns.

The objective is to learn how to reconstruct normal observations. Observations that produce unusually large reconstruction errors may be considered anomalous, subject to threshold calibration and evaluation.

USAD will be evaluated as a separate model from the LSTM and TCN Autoencoders. Its performance will not be assumed to be superior simply because it uses a different architecture.

### 7.4.2 — Architecture

USAD consists of three neural-network components:

- **Encoder, \(E\):** Maps an input observation to a lower-dimensional latent representation.
- **Decoder 1, \(D_1\):** Reconstructs the input from the latent representation.
- **Decoder 2, \(D_2\):** Provides a second reconstruction path and participates in the joint training objective.

For an input \(x\), the first reconstruction is

$$
w_1 = D_1(E(x)).
$$

The second reconstruction is

$$
w_2 = D_2(E(x)).
$$

A third reconstruction is obtained by passing the first reconstruction through the encoder and second decoder:

$$
w_3 = D_2(E(w_1)).
$$

The two decoders provide complementary reconstruction objectives. The training procedure encourages the first decoder to reconstruct the input while the second decoder learns through a competing reconstruction objective.

### 7.4.3 — Mathematical Objective

Let the reconstruction loss be the mean squared error (MSE):

$$
\operatorname{MSE}(x,w)
=
\frac{1}{d}\sum_{j=1}^{d}(x_j-w_j)^2,
$$

where \(d\) is the input dimensionality, \(x_j\) is the \(j\)-th input value, and \(w_j\) is its reconstructed value.

USAD uses two losses. At training epoch \(n\), define

$$
\alpha_n = \frac{1}{n},
\qquad
\beta_n = 1-\frac{1}{n}.
$$

The first objective is

$$
\mathcal{L}_1
=
\alpha_n\operatorname{MSE}(x,w_1)
+
\beta_n\operatorname{MSE}(x,w_3).
$$

The second objective is

$$
\mathcal{L}_2
=
\alpha_n\operatorname{MSE}(x,w_2)
-
\beta_n\operatorname{MSE}(x,w_3).
$$

The first objective minimizes reconstruction errors along the first and third paths. The second combines reconstruction of the input by the second decoder with a competing objective on the third reconstruction.

The two objectives are optimized using separate optimizer updates, with the encoder participating in both updates. This dual-objective training procedure is a defining feature of USAD and must be implemented correctly.

### 7.4.4 — Temporal Input Representation

The existing dataset is represented as multivariate sequences with 30 timesteps and 42 features.

For the initial USAD implementation, each sequence will be flattened into a single vector:

$$
x \in \mathbb{R}^{30\times42}
\quad\longrightarrow\quad
x_{\mathrm{flat}} \in \mathbb{R}^{1260}.
$$

Flattening preserves the values and their ordering within each window, allowing a fully connected USAD architecture to learn patterns across the complete window.

Unlike the TCN, this representation does not explicitly use convolutional layers. Temporal order is represented by the fixed positions of the flattened features.

The input and output dimensions must match, and no sequence may cross the established training, monitoring, threshold-calibration, or validation boundaries.

### 7.4.5 — Training and Evaluation Protocol

The preliminary USAD experiment will follow the shared experimental protocol established earlier in this notebook.

- Use the existing training sequences and the scaler fitted on training data only.
- Use the monitoring partition for checkpoint selection and training decisions.
- Preserve chronological partition boundaries.
- Do not use threshold-calibration or final-evaluation observations to select model parameters.
- Save the model checkpoint, training history, configuration metadata, and learning curves to persistent project storage.
- Reserve systematic hyperparameter tuning for Notebook 08.
- Calibrate the anomaly threshold only after model selection.

Reconstruction error measures how closely the model reproduces an input; it does not directly establish whether that input represents an attack or an operationally significant anomaly.

The preliminary experiment will therefore document reconstruction performance without claiming attack-detection effectiveness or declaring USAD the best model.

In [ ]:
import torch.nn as nn

# 2. Define the preliminary USAD architecture

USAD_SEQUENCE_LENGTH = X_train_seq.shape[1]
USAD_FEATURE_COUNT = X_train_seq.shape[2]
USAD_INPUT_DIM = USAD_SEQUENCE_LENGTH * USAD_FEATURE_COUNT

# Preliminary architecture; tuning belongs in Notebook 08.
USAD_HIDDEN_DIM = 128
USAD_LATENT_DIM = 32


class USAD(nn.Module):
    """
    Fully connected USAD architecture.

    Input:
        Flattened sequence with shape (batch_size, input_dim).

    Outputs:
        w1: Decoder 1 reconstruction.
        w2: Decoder 2 reconstruction.
        w3: Decoder 2 reconstruction of w1.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        latent_dim: int,
    ):
        super().__init__()

        if min(input_dim, hidden_dim, latent_dim) <= 0:
            raise ValueError("All architecture dimensions must be positive.")

        # Shared encoder.
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )

        # First decoder.
        self.decoder1 = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

        # Second decoder.
        self.decoder2 = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        """
        Return the three reconstruction paths.
        """
        latent = self.encoder(x)

        w1 = self.decoder1(latent)
        w2 = self.decoder2(latent)

        # Re-encode the first reconstruction, then decode it
        # through Decoder 2.
        w3 = self.decoder2(self.encoder(w1))

        return w1, w2, w3



In [ ]:
import copy
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset

required_objects = [
    "X_train_seq",
    "X_monitor_seq",
    "USAD",
    "USAD_INPUT_DIM",
    "USAD_HIDDEN_DIM",
    "USAD_LATENT_DIM",
    "DEVICE",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Required objects are missing: {missing_objects}. "
        "Run the preceding data-preparation and architecture cells."
    )

assert X_train_seq.ndim == 3
assert X_monitor_seq.ndim == 3
assert X_train_seq.shape[1:] == X_monitor_seq.shape[1:]
assert X_train_seq.shape[1:] == (30, 42)
assert USAD_INPUT_DIM == 1260

if not np.isfinite(X_train_seq).all():
    raise ValueError("Training sequences contain non-finite values.")

if not np.isfinite(X_monitor_seq).all():
    raise ValueError("Monitoring sequences contain non-finite values.")

if len(X_train_seq) == 0 or len(X_monitor_seq) == 0:
    raise ValueError("Training and monitoring partitions must be non-empty.")

DEVICE = torch.device(DEVICE)

if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA was requested but is unavailable. "
        "Check the runtime before proceeding."
    )

In [ ]:
# Learning rate is reduced 10x from the earlier attempt (1e-3 -> 1e-4)
# and gradient clipping is added. Both changes target the divergence
# observed in the previous run, where L2's adversarial term on w3
# produced catastrophic parameter updates by epoch 2.

USAD_SEED = 42
USAD_BATCH_SIZE = 256
USAD_MAX_EPOCHS = 50
USAD_EARLY_STOPPING_PATIENCE = 5
USAD_LEARNING_RATE = 1e-4
USAD_MAX_GRAD_NORM = 1.0

random.seed(USAD_SEED)
np.random.seed(USAD_SEED)
torch.manual_seed(USAD_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(USAD_SEED)

# Reinitialize the model after setting the random seed.
usad_model = USAD(
    input_dim=USAD_INPUT_DIM,
    hidden_dim=USAD_HIDDEN_DIM,
    latent_dim=USAD_LATENT_DIM,
).to(DEVICE)

# Two optimizers share the encoder, each owns one decoder.
# The encoder is updated twice per batch — this is the USAD
# training procedure, not a bug.
optimizer1 = torch.optim.Adam(
    list(usad_model.encoder.parameters())
    + list(usad_model.decoder1.parameters()),
    lr=USAD_LEARNING_RATE,
)

optimizer2 = torch.optim.Adam(
    list(usad_model.encoder.parameters())
    + list(usad_model.decoder2.parameters()),
    lr=USAD_LEARNING_RATE,
)

In [ ]:
# Loaders share the existing NumPy arrays rather than copying them.
# Sequences are flattened per batch, not all at once.

train_tensor = torch.from_numpy(X_train_seq)
monitor_tensor = torch.from_numpy(X_monitor_seq)

train_dataset = TensorDataset(train_tensor)
monitor_dataset = TensorDataset(monitor_tensor)

loader_generator = torch.Generator()
loader_generator.manual_seed(USAD_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=USAD_BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
    generator=loader_generator,
)

monitor_loader = DataLoader(
    monitor_dataset,
    batch_size=USAD_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)

In [ ]:
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/swat-artifacts/usad_preliminary"
)

if not Path("/content/drive/MyDrive").is_dir():
    raise FileNotFoundError(
        "Google Drive is not mounted at /content/drive/MyDrive. "
        "Mount Drive before running this cell."
    )

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = ARTIFACT_DIR / "usad_best.pth"
HISTORY_PATH = ARTIFACT_DIR / "usad_training_history.csv"
METADATA_PATH = ARTIFACT_DIR / "usad_training_metadata.json"
MSE_PLOT_PATH = ARTIFACT_DIR / "usad_training_mse.png"
MAE_PLOT_PATH = ARTIFACT_DIR / "usad_training_mae.png"

In [ ]:
def usad_losses(x, w1, w2, w3, epoch_number):
    """
    Calculate the two USAD training objectives.

    alpha = 1 / epoch
    beta  = 1 - alpha

    L1 = alpha * MSE(x, w1) + beta * MSE(x, w3)
    L2 = alpha * MSE(x, w2) - beta * MSE(x, w3)
    """
    alpha = 1.0 / epoch_number
    beta = 1.0 - alpha

    mse1 = F.mse_loss(w1, x)
    mse2 = F.mse_loss(w2, x)
    mse3 = F.mse_loss(w3, x)

    loss1 = alpha * mse1 + beta * mse3
    loss2 = alpha * mse2 - beta * mse3

    return loss1, loss2


def evaluate_usad(model, data_loader, device):
    """
    Evaluate direct reconstruction errors on the monitoring data.

    The checkpoint-selection score is:
        0.5 * (MSE(x, w1) + MSE(x, w2))

    This is a reconstruction score, not an attack-detection metric.
    """
    model.eval()

    total_mse1 = 0.0
    total_mse2 = 0.0
    total_mae1 = 0.0
    total_mae2 = 0.0
    total_count = 0

    with torch.no_grad():
        for (sequences,) in data_loader:

            batch_size = sequences.shape[0]

            x = sequences.to(
                device=device,
                dtype=torch.float32,
                non_blocking=True,
            ).reshape(batch_size, -1)

            w1, w2, _ = model(x)

            if not (
                torch.isfinite(w1).all()
                and torch.isfinite(w2).all()
            ):
                raise FloatingPointError(
                    "USAD produced non-finite reconstruction "
                    "values during monitoring."
                )

            mse1 = F.mse_loss(w1, x, reduction="none").mean(dim=1)
            mse2 = F.mse_loss(w2, x, reduction="none").mean(dim=1)
            mae1 = F.l1_loss(w1, x, reduction="none").mean(dim=1)
            mae2 = F.l1_loss(w2, x, reduction="none").mean(dim=1)

            total_mse1 += mse1.sum().item()
            total_mse2 += mse2.sum().item()
            total_mae1 += mae1.sum().item()
            total_mae2 += mae2.sum().item()

            total_count += batch_size

    if total_count == 0:
        raise ValueError("The monitoring loader contains no observations.")

    metrics = {
        "monitor_mse_w1": total_mse1 / total_count,
        "monitor_mse_w2": total_mse2 / total_count,
        "monitor_mae_w1": total_mae1 / total_count,
        "monitor_mae_w2": total_mae2 / total_count,
    }

    metrics["monitor_reconstruction_mse"] = 0.5 * (
        metrics["monitor_mse_w1"] + metrics["monitor_mse_w2"]
    )

    metrics["monitor_reconstruction_mae"] = 0.5 * (
        metrics["monitor_mae_w1"] + metrics["monitor_mae_w2"]
    )

    if not np.isfinite(list(metrics.values())).all():
        raise FloatingPointError(
            "USAD monitoring metrics contain non-finite values."
        )

    return metrics

In [ ]:
def parameters_are_finite(model):
    return all(
        torch.isfinite(parameter).all().item()
        for parameter in model.parameters()
    )


def gradients_are_finite(parameters):
    return all(
        parameter.grad is None
        or torch.isfinite(parameter.grad).all().item()
        for parameter in parameters
    )


def clip_and_validate_gradients(parameters, max_norm, label):
    """
    Check gradients before and after clipping.
    Returns the gradient norm before clipping.
    """
    parameters = list(parameters)

    if not gradients_are_finite(parameters):
        raise FloatingPointError(
            f"Non-finite gradients detected during {label}."
        )

    grad_norm = torch.nn.utils.clip_grad_norm_(
        parameters,
        max_norm=max_norm,
        error_if_nonfinite=True,
    )

    if not torch.isfinite(grad_norm):
        raise FloatingPointError(
            f"Non-finite gradient norm during {label}."
        )

    if not gradients_are_finite(parameters):
        raise FloatingPointError(
            f"Non-finite clipped gradients during {label}."
        )

    return float(grad_norm.detach().item())

In [ ]:
history = []

best_monitor_mse = float("inf")
best_epoch = None
best_state_dict = None

epochs_without_improvement = 0
training_start = time.perf_counter()

stop_reason = "Maximum epoch count reached."
training_error = None

print("Starting stabilized preliminary USAD training.")
print(f"Training sequences:     {len(X_train_seq):,}")
print(f"Monitoring sequences:   {len(X_monitor_seq):,}")
print(f"Sequence shape:         {X_train_seq.shape[1:]}")
print(f"Input dimension:        {USAD_INPUT_DIM:,}")
print(f"Hidden dimension:       {USAD_HIDDEN_DIM}")
print(f"Latent dimension:       {USAD_LATENT_DIM}")
print(f"Batch size:             {USAD_BATCH_SIZE}")
print(f"Maximum epochs:         {USAD_MAX_EPOCHS}")
print(f"Learning rate:          {USAD_LEARNING_RATE}")
print(f"Gradient clipping norm: {USAD_MAX_GRAD_NORM}")
print(f"Device:                 {DEVICE}")

for epoch in range(1, USAD_MAX_EPOCHS + 1):

    usad_model.train()

    epoch_loss1_sum = 0.0
    epoch_loss2_sum = 0.0
    epoch_grad_norm1_sum = 0.0
    epoch_grad_norm2_sum = 0.0
    processed_sequences = 0
    epoch_start = time.perf_counter()

    try:
        for (sequences,) in train_loader:

            batch_size = sequences.shape[0]

            x = sequences.to(
                device=DEVICE,
                dtype=torch.float32,
                non_blocking=True,
            ).reshape(batch_size, -1)

            # Optimizer 1: encoder + decoder 1.
            optimizer1.zero_grad(set_to_none=True)
            optimizer2.zero_grad(set_to_none=True)

            w1, w2, w3 = usad_model(x)

            loss1, _ = usad_losses(x, w1, w2, w3, epoch)

            if not torch.isfinite(loss1).item():
                raise FloatingPointError(
                    f"Non-finite L1 loss at epoch {epoch}."
                )

            loss1.backward()

            params1 = (
                list(usad_model.encoder.parameters())
                + list(usad_model.decoder1.parameters())
            )

            grad_norm1 = clip_and_validate_gradients(
                params1,
                max_norm=USAD_MAX_GRAD_NORM,
                label="optimizer 1",
            )

            optimizer1.step()

            if not parameters_are_finite(usad_model):
                raise FloatingPointError(
                    "Non-finite model parameters after optimizer 1 "
                    f"at epoch {epoch}."
                )

            # Optimizer 2: encoder + decoder 2.
            # Recompute after optimizer 1 changes the weights.
            optimizer1.zero_grad(set_to_none=True)
            optimizer2.zero_grad(set_to_none=True)

            w1_second, w2_second, w3_second = usad_model(x)

            _, loss2 = usad_losses(
                x, w1_second, w2_second, w3_second, epoch
            )

            if not torch.isfinite(loss2).item():
                raise FloatingPointError(
                    f"Non-finite L2 loss at epoch {epoch}."
                )

            loss2.backward()

            params2 = (
                list(usad_model.encoder.parameters())
                + list(usad_model.decoder2.parameters())
            )

            grad_norm2 = clip_and_validate_gradients(
                params2,
                max_norm=USAD_MAX_GRAD_NORM,
                label="optimizer 2",
            )

            optimizer2.step()

            if not parameters_are_finite(usad_model):
                raise FloatingPointError(
                    "Non-finite model parameters after optimizer 2 "
                    f"at epoch {epoch}."
                )

            epoch_loss1_sum += loss1.detach().item() * batch_size
            epoch_loss2_sum += loss2.detach().item() * batch_size
            epoch_grad_norm1_sum += grad_norm1
            epoch_grad_norm2_sum += grad_norm2
            processed_sequences += batch_size

    except (FloatingPointError, RuntimeError) as exc:
        training_error = str(exc)
        stop_reason = (
            "Training stopped after a numerical-stability error."
        )
        print(f"\nTRAINING STOPPED: {training_error}")
        break

    if processed_sequences != len(train_dataset):
        raise RuntimeError(
            "The number of processed training sequences does not "
            "match the training dataset."
        )

    mean_train_loss1 = epoch_loss1_sum / processed_sequences
    mean_train_loss2 = epoch_loss2_sum / processed_sequences
    mean_grad_norm1 = epoch_grad_norm1_sum / len(train_loader)
    mean_grad_norm2 = epoch_grad_norm2_sum / len(train_loader)

    if not np.isfinite([
        mean_train_loss1,
        mean_train_loss2,
        mean_grad_norm1,
        mean_grad_norm2,
    ]).all():
        training_error = (
            f"Non-finite training summary at epoch {epoch}."
        )
        stop_reason = (
            "Training stopped after a numerical-stability error."
        )
        print(f"\nTRAINING STOPPED: {training_error}")
        break

    try:
        monitor_metrics = evaluate_usad(
            usad_model, monitor_loader, DEVICE
        )
    except (FloatingPointError, RuntimeError) as exc:
        training_error = str(exc)
        stop_reason = "Training stopped after a monitoring error."
        print(f"\nTRAINING STOPPED: {training_error}")
        break

    monitor_mse = monitor_metrics["monitor_reconstruction_mse"]
    epoch_seconds = time.perf_counter() - epoch_start

    row = {
        "epoch": epoch,
        "train_loss1": mean_train_loss1,
        "train_loss2": mean_train_loss2,
        "mean_grad_norm1_before_clipping": mean_grad_norm1,
        "mean_grad_norm2_before_clipping": mean_grad_norm2,
        **monitor_metrics,
        "epoch_seconds": epoch_seconds,
    }

    history.append(row)

    improved = monitor_mse < best_monitor_mse

    if improved:
        best_monitor_mse = monitor_mse
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state_dict = {
            name: tensor.detach().cpu().clone()
            for name, tensor in usad_model.state_dict().items()
        }

        checkpoint = {
            "model_state_dict": best_state_dict,
            "best_epoch": best_epoch,
            "best_monitor_reconstruction_mse": best_monitor_mse,
            "config": {
                "seed": USAD_SEED,
                "sequence_length": int(X_train_seq.shape[1]),
                "feature_count": int(X_train_seq.shape[2]),
                "input_dim": USAD_INPUT_DIM,
                "hidden_dim": USAD_HIDDEN_DIM,
                "latent_dim": USAD_LATENT_DIM,
                "batch_size": USAD_BATCH_SIZE,
                "max_epochs": USAD_MAX_EPOCHS,
                "learning_rate": USAD_LEARNING_RATE,
                "max_gradient_norm": USAD_MAX_GRAD_NORM,
                "early_stopping_patience": (
                    USAD_EARLY_STOPPING_PATIENCE
                ),
            },
        }

        torch.save(checkpoint, CHECKPOINT_PATH)

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d}/{USAD_MAX_EPOCHS} | "
        f"L1: {mean_train_loss1:.7f} | "
        f"L2: {mean_train_loss2:.7f} | "
        f"Monitor MSE: {monitor_mse:.7f} | "
        f"Monitor MAE: "
        f"{monitor_metrics['monitor_reconstruction_mae']:.7f} | "
        f"Grad norms: {mean_grad_norm1:.4f}, "
        f"{mean_grad_norm2:.4f} | "
        f"Best epoch: {best_epoch:02d} | "
        f"{'BEST' if improved else ''} | "
        f"{epoch_seconds:.1f}s"
    )

    # Persist the history after each completed epoch.
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    if epochs_without_improvement >= USAD_EARLY_STOPPING_PATIENCE:
        stop_reason = (
            "Early stopping: monitoring reconstruction MSE "
            "did not improve for "
            f"{USAD_EARLY_STOPPING_PATIENCE} consecutive epochs."
        )
        print(f"\n{stop_reason}")
        break

In [ ]:
training_seconds = time.perf_counter() - training_start

if best_state_dict is not None:

    usad_model.load_state_dict(best_state_dict)
    usad_model.to(DEVICE)
    usad_model.eval()

    restored_metrics = evaluate_usad(
        usad_model, monitor_loader, DEVICE
    )

    restored_mse = restored_metrics["monitor_reconstruction_mse"]

    if not np.isclose(
        restored_mse,
        best_monitor_mse,
        rtol=1e-5,
        atol=1e-8,
    ):
        raise RuntimeError(
            "Restored checkpoint monitoring MSE does not match "
            "the recorded best monitoring MSE."
        )

else:
    restored_metrics = None

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

metadata = {
    "experiment": "USAD preliminary stabilization run",
    "seed": USAD_SEED,
    "device": str(DEVICE),
    "train_sequences": int(len(X_train_seq)),
    "monitor_sequences": int(len(X_monitor_seq)),
    "sequence_length": int(X_train_seq.shape[1]),
    "feature_count": int(X_train_seq.shape[2]),
    "input_dim": int(USAD_INPUT_DIM),
    "hidden_dim": int(USAD_HIDDEN_DIM),
    "latent_dim": int(USAD_LATENT_DIM),
    "batch_size": USAD_BATCH_SIZE,
    "max_epochs": USAD_MAX_EPOCHS,
    "learning_rate": USAD_LEARNING_RATE,
    "max_gradient_norm": USAD_MAX_GRAD_NORM,
    "early_stopping_patience": USAD_EARLY_STOPPING_PATIENCE,
    "epochs_completed": int(len(history)),
    "best_epoch": int(best_epoch) if best_epoch is not None else None,
    "best_monitor_reconstruction_mse": (
        float(best_monitor_mse) if best_epoch is not None else None
    ),
    "training_seconds": float(training_seconds),
    "stop_reason": stop_reason,
    "training_error": training_error,
    "best_checkpoint_restored": best_state_dict is not None,
    "restored_monitor_metrics": restored_metrics,
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)


if not history_df.empty:

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        history_df["epoch"],
        history_df["train_loss1"],
        label="Training L1",
    )
    ax.plot(
        history_df["epoch"],
        history_df["train_loss2"],
        label="Training L2",
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("USAD Training Losses")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(MSE_PLOT_PATH, dpi=150)
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(
        history_df["epoch"],
        history_df["monitor_reconstruction_mae"],
        label="Monitoring reconstruction MAE",
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MAE")
    ax.set_title("USAD Monitoring Reconstruction MAE")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(MAE_PLOT_PATH, dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
print("USAD PRELIMINARY TRAINING SUMMARY")
print(f"Epochs completed: {len(history)}")
print(f"Training duration: {training_seconds / 60:.2f} minutes")
print(f"Stop reason: {stop_reason}")

if training_error is not None:
    print(f"Training error: {training_error}")

if best_epoch is not None:
    print(f"Best epoch: {best_epoch}")
    print(
        "Best monitoring reconstruction MSE: "
        f"{best_monitor_mse:.8f}"
    )
    print(
        "Restored monitoring reconstruction MAE: "
        f"{restored_metrics['monitor_reconstruction_mae']:.8f}"
    )
else:
    print("No valid checkpoint was produced.")

print("\nArtifact verification:")

for path in [
    CHECKPOINT_PATH,
    HISTORY_PATH,
    METADATA_PATH,
    MSE_PLOT_PATH,
    MAE_PLOT_PATH,
]:
    exists = path.is_file()
    size = path.stat().st_size if exists else 0
    print(
        f"{path.name}: "
        f"{'OK' if exists and size > 0 else 'MISSING OR EMPTY'}"
        f" ({size:,} bytes)"
    )

if best_state_dict is None:
    print(
        "\nRESULT: USAD training did not produce a valid checkpoint. "
        "Do not proceed to model evaluation with this run."
    )
elif training_error is not None:
    print(
        "\nRESULT: A checkpoint was restored, but training stopped "
        "because of a numerical-stability error. Treat this run "
        "as a diagnostic failure, not a successful training run."
    )
else:
    print(
        "\nRESULT: A finite best checkpoint was restored. "
        "Review the training history before deciding whether "
        "the run is sufficiently stable for the next stage."
    )